# mini-beatrix-2 — the governed full-splat mission

**Craft**: `mini-beatrix-2` — d1024 × L32, ctx 8192, a governed CausalSplatHUB in **every** block
(16 constellations × 256 anchors @ D=256 each, supply 1.0×D), 6 banks @ ff1024, ~849M params.
Plan of record: claude-mind `history/plans/2026-08-26_mini_beatrix_v2_shape.md` · sizing console: Beatrix Foundry.

**Laws embodied** (all measured): supply K ≤ 2·D per book (ROUND 5e) · anchor governor from birth
(min-sep projection, post-step, identity-when-slack — ROUND 5f) · product-code composition by budget,
never softmax over books (B4) · pure Adam + Muon split, wd 0 · bf16-train / fp8-ship / never-fp16
(squared address terms flush) · crash-safe resume-first · **per-boundary report + checkpoint pushes
(the ship-complete law — no training without push cells)**.

**Binding rider**: the additive hub write saturates at high demand (R=64: .84 vs .99). The delta-write
screen is queued separately; until it passes, no claim here covers retrieval.

Target hardware: RTX 6000 Pro Blackwell 96GB. `TURBO` stays False — compiled bf16 backward NaNs on
Blackwell (measured); eager-fused is the training path.

Recommended first session: run Cell P once with `PRESET='mini-beatrix-2s'` (the 237M screen craft) —
the gating cells (full-splat causal viability at 2ep-scale budgets) are cheapest there.

In [ ]:
# C1 — pinned installs. RESTART RUNTIME if the version line below changes.
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # Linux Colab: kills the fragmentation tax
%pip install -q "datasets>=5.0.0" "geolip-alephllm @ git+https://github.com/AbstractEyes/alephllm@af9319ced49866ec563d611cd14c17874bcf619c"
import geolip.alephllm as A
print('geolip.alephllm', A.__version__)
assert tuple(int(x) for x in A.__version__.split('.')) >= (0, 8, 7), (
    'RESTART REQUIRED: runtime still holds the old package — Runtime > Restart, then rerun from C1')

# THE ARCHITECTURE CONTROL (Law E, 2026-08-31): the pure-sdpa twin —
# hub layers removed, everything else the treatment's BIRTH recipe
# verbatim (born-null unfrozen head; same curriculum, seeds, anneal
# flow). This run is what makes the full-splat +3.4 bpb claim a
# comparison instead of a self-report. Own hub prefix on the hub.
CRAFT = 'mini-beatrix-2s-control'


In [ ]:
# C2 (P) — PREFLIGHT: build, law check, bit-exact births, bench gate. No training.
import time, torch
from google.colab import userdata
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.model.governor import govern_model

HF_TOKEN = userdata.get('HF_TOKEN')
PRESET = CRAFT          # 'mini-beatrix-2s' for the screen craft; '-control' twins exist
p = get_preset(PRESET); cfg = p.model; tc = p.train
torch.manual_seed(tc.seed)
model = AlephLM(cfg).cuda()
n = model.param_count()
print(f'{cfg.name}: {n/1e6:.1f}M params | hubs {len(cfg.hub_layers)}/{cfg.n_layers} blocks × '
      f'{cfg.hub_const} books × {cfg.hub_K}@D{cfg.hub_D} (supply {cfg.hub_K/cfg.hub_D:.2f}×D) | ctx {cfg.context}')
assert cfg.hub_K <= 2 * cfg.hub_D, 'supply law violated — this preset should not exist'
hits = govern_model(model, tc.governor_theta)
print(f'governor birth check: {hits} hits (must be 0 at mission D)'); assert hits == 0
x = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
with torch.no_grad():   # C6 null paths: bank + head aleph contribute exactly zero at init
    a = model(x, disable_bank=True, disable_head_aleph=True).logits
    b = model(x).logits
    assert torch.equal(a, b), 'C6 violated — born-null paths are not null'
print('C6 null paths bit-exact at birth')
opt_probe = torch.optim.SGD(model.parameters(), lr=0.0)
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
for i in range(3):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        out = model(x, targets=x)
    out.loss.backward(); opt_probe.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sps = (time.time() - t0) / 3
vram = torch.cuda.max_memory_allocated() / 2**30
toks = tc.micro_batch * cfg.context
print(f'bench: {sps:.2f}s / micro-step · {toks/sps/1e3:.0f}k tok/s (×{tc.grad_accum} accum) · peak {vram:.1f} GB')
assert vram < 88, f'over the Blackwell budget at micro_batch={tc.micro_batch} — lower it in the preset'
del model, out, opt_probe; torch.cuda.empty_cache()
print('PREFLIGHT PASS — Cell T is armed')

In [ ]:
# C3 (T) — THE SESSION. Resume-first; stops at stage boundaries; every boundary
# ships a checkpoint (trainer HubSync) AND a report JSON (this cell). Interrupt-safe.
import gc, io, json, time, torch
from google.colab import userdata
from geolip.alephllm import prepare
from geolip.alephllm.data import curriculum as C
from geolip.alephllm.train import probes
from geolip.alephllm.train.instruments import model_census, toggle_ledger, readout, special_token_gauge
from geolip.alephllm.model.governor import govern_model

PRESET = CRAFT
MAX_HOURS = 10.5

run = prepare(PRESET, hf_token=userdata.get('HF_TOKEN'))
C.append_curriculum_phases(run.manifest)            # idempotent: adds S0–S8 after the pretrain phases

def boundary_report(run, tag):
    m, tok, dev = run.raw_model, run.tokenizer, run.device
    m.eval()
    with torch.no_grad():
        pr = probes.run_all(m, tok, dev)
        sample = torch.randint(0, 255, (2, min(1024, m.cfg.context)), device=dev)
        census = model_census(m, sample)
        # the specials + ledger gauges belong IN the boundary JSON
        # (instrument-first): C4 charts doc/reset bpb per stage
        ledger = toggle_ledger(m, run._val())
        sp = special_token_gauge(m, run._val())
    rep = {'tag': tag, 'step': run.step, 'tokens': run.manifest.tokens_seen,
           'probes': pr, 'census_flags': census.get('flags'),
           'ledger': ledger, 'special': sp,
           'governor_hits_cum': getattr(run, '_gov_hits', 0),
           'crowd_check_extra_hits': govern_model(m, run.tc.governor_theta) if run.tc.governor else None}
    run.hub.upload_bytes(json.dumps(rep, default=float).encode(),
                         f'reports/v2/{tag}_step{run.step}.json')
    print(probes.report(pr)); m.train()
    return rep

# 0.8.1 driver contract: train(stop_at_boundary=True) RETURNS at every
# phase boundary with run._last_boundary = the phase that just finished;
# run._interrupted = manual stop (NEVER auto-resume one - the run0 driver
# silently resumed an interrupt: audit catch 2026-08-26).
t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    hours_left = (t_end - time.time()) / 3600
    if hours_left < 0.2: break
    run.train(max_hours=hours_left, stop_at_boundary=True)
    if getattr(run, '_interrupted', False):
        print('manual stop - no auto-resume; resume state is on the hub')
        break
    tag = getattr(run, '_last_boundary', None) or 'session_cap'
    boundary_report(run, tag)
    if run.manifest.current_phase() is None:
        print('curriculum complete'); break
    gc.collect(); torch.cuda.empty_cache()
print('session over — resume state on the hub')

In [ ]:
# C6 — ACTIVATE THE TWO-PHASE ANNEAL. Run ONLY after 'curriculum complete'.
# Phil 2026-08-31: "enable the anneal mix without chat and the anneal mix
# with chat, we'll run without first, then just train chat on after."
# Phase 1 anneal_nochat: the mix MINUS the chat frame (renormalized) — its
# boundary report is the exact pre-chat baseline. Phase 2 anneal_mix: the
# full mix WITH the chat frame (first-ever gradient on SYS/USER/MODEL/END).
import gc, time, torch
assert run.manifest.current_phase() is None, 'curriculum still active'
phases = run.manifest.phases
anneal = next(ph for ph in phases if ph['name'] == 'anneal_mix')
anneal['status'] = 'planned'
anneal['planned_tokens'] = 1_000_000_000
if not any(ph['name'] == 'anneal_nochat' for ph in phases):
    phases.insert(phases.index(anneal), dict(
        name='anneal_nochat', dataset='anneal-nochat',
        planned_tokens=1_000_000_000, status='planned'))
run.manifest.note('two-phase anneal activated: nochat 1B then chat 1B '
                  '(plan of record 2026-08-31)')
print(run.status())
t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    hours_left = (t_end - time.time()) / 3600
    if hours_left < 0.2: break
    run.train(max_hours=hours_left, stop_at_boundary=True)
    if getattr(run, '_interrupted', False):
        print('manual stop - no auto-resume'); break
    tag = getattr(run, '_last_boundary', None) or 'session_cap'
    boundary_report(run, tag)
    if run.manifest.current_phase() is None:
        print('anneal complete - the mission curriculum is FINISHED'); break
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# C4 — growth table from the boundary reports. (Fixed 2026-08-31: probe
# values are DICTS ({'acc': ...}) — the old float filter printed empty
# columns; and op reports (revival_*) carry no 'probes' — skip, don't crash.)
import json
from huggingface_hub import HfApi, hf_hub_download
from geolip.alephllm.presets import TRAINING_REPO
PRESET = CRAFT
api = HfApi()
files = sorted(f for f in api.list_repo_files(TRAINING_REPO)
               if f.startswith(f'{PRESET}/reports/v2/') and f.endswith('.json'))
rows = []
for f in files:
    r = json.load(open(hf_hub_download(TRAINING_REPO, f)))
    pr = r.get('probes')
    if not pr:
        continue                       # op/provenance reports have none
    accs = {k.split('_', 1)[0]: round(v.get('acc', v) if isinstance(v, dict)
                                      else v, 2) for k, v in pr.items()}
    led = r.get('ledger') or {}
    sp = r.get('special') or {}
    rows.append((r['tag'][:14], r['step'], round(r['tokens'] / 1e9, 2),
                 led.get('bpb_full'), led.get('toggle_head_aleph_off'),
                 sp.get('doc_bpb'), accs, r.get('governor_hits_cum')))
rows.sort(key=lambda x: x[1])
hdr = f"{'stage':<15}{'step':>7}{'tok(B)':>7}{'val':>7}{'head':>8}{'doc':>6}  probes"
print(hdr)
for tag, step, tk, val, hd, doc, accs, gh in rows:
    v = f'{val:.3f}' if isinstance(val, float) else '  --'
    h = f'{hd:+.3f}' if isinstance(hd, float) else '    --'
    d = f'{doc:.1f}' if isinstance(doc, float) else '  --'
    print(f'{tag:<15}{step:>7}{tk:>7}{v:>7}{h:>8}{d:>6}  {accs}')

### Notes
- **Screen first**: `mini-beatrix-2s` runs every gating cell at 3.6× less cost; the big craft inherits verdicts, not hopes.
- **Controls**: `mini-beatrix-2-control` (pure sdpa) and `governor=''` (set on a copied preset) are the two arms any v2 claim must beat/tie under Law E bars (2 seeds; the seed field lives in `TrainConfig.seed`).
- **Monitoring keys**: `governor/hits_cum` (TB) — expect 0 early, small and rising only if crowding pressure exists; `hub_addr.anchor_merge_pairs` must stay 0 (the governor's whole job); `hub_consumed_erank` per block is the supply-utilization gauge (v1's L4 read 2 — the disease this design retires).
- **Kernels**: eager-fused path only on Blackwell (compiled bf16 backward NaNs — measured); `hub_chunk=256`; fp16 forbidden.
- Sessions are boundary-exact: interrupting mid-stage is safe (resume-first), but reports only ship at boundaries.